In [15]:
# %% ── Cell 1: CONFIG ─────────────────────────────────────────────────────────

CONFIG = {
    # ── Input ──────────────────────────────────────────────────────────────
    "h5ad_path":    r"C:\Users\ulago\Downloads\Variant_Vax_obj.h5ad",
    "go_csv":       None,          # set to saved GMT pkl path to skip download

    # ── Column names (None = auto-detect) ──────────────────────────────────
    "groupby":      None,          # cell-type / cluster column
    "severity_col": "WHO_Score_at_Peak",          # severity / outcome column
    "severity_type":"categorical",
    "PILOT_MODE": True,
    "pilot_fraction": 0.2,            # fraction of data to use in pilot mode (for quick testing)
    "pilot_seed": 42,                 # random seed for pilot mode sampling
    "stratify_by": None, # "categorical" or "continuous"

    # ── HVG filtering (speeds up mask building) ────────────────────────────
    "use_hvg":      True,
    "n_hvg":        5000,

    # ── GO mask ────────────────────────────────────────────────────────────
    "gmt_libraries": [
        "GO_Biological_Process_2023",
        "GO_Molecular_Function_2023",
        "GO_Cellular_Component_2023",
    ],
    "gmt_cache_dir":         "gmt_cache",
    "min_genes_per_pathway": 5,
    "max_genes_per_pathway": 500,  # tighter than before — removes generic terms

    # ── Pathway scoring ────────────────────────────────────────────────────
    # Method: mean z-scored expression of pathway member genes per cell
    # (fast, no GPU, interpretable, comparable to ssGSEA)
    "score_method": "mean_z",      # "mean_z" | "mean_raw" | "pca1"

    # ── ML models ──────────────────────────────────────────────────────────
    "test_fraction":  0.20,
    "random_seed":    42,
    "xgb_n_estimators": 300,
    "xgb_max_depth":    4,
    "xgb_lr":          0.05,
    "rf_n_estimators":  200,
    "enet_l1_ratio":    0.5,       # 0=Ridge, 1=Lasso, 0.5=ElasticNet
    "n_shap_cells":     500,       # cells to explain with SHAP

    # ── UMAP ───────────────────────────────────────────────────────────────
    "umap_n_neighbors": 15,
    "umap_min_dist":    0.3,
    "umap_metric":      "cosine",  # cosine works well for sparse pathway vectors

    # ── Output ─────────────────────────────────────────────────────────────
    "outdir": "severity_results",
}
# Training overrides applied automatically in pilot mode
PILOT_OVERRIDES = {
    "epochs":     20,    # enough to see whether loss converges
    "batch_size": 128,   # smaller batches for smaller dataset
}
 
FULL_OVERRIDES = {}      # no overrides for the full run

In [16]:
# ── Apply pilot / full overrides ─────────────────────────────────────────────
if CONFIG["PILOT_MODE"]:
    CONFIG.update(PILOT_OVERRIDES)
    RUN_TAG  = "pilot"
    print(f"PILOT MODE  — {int(CONFIG['pilot_fraction']*100)}% subsample, "
          f"{CONFIG['epochs']} epochs, batch {CONFIG['batch_size']}")
else:
    CONFIG.update(FULL_OVERRIDES)
    RUN_TAG  = "full"
    print(f"FULL RUN MODE — all cells, {CONFIG['epochs']} epochs")
 
OUTDIR = Path(CONFIG["outdir"]) / RUN_TAG
OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir  : {OUTDIR.resolve()}")

PILOT MODE  — 20% subsample, 20 epochs, batch 128
Output dir  : C:\Users\ulago\Downloads\Applied-Machine-Learning-Final-Project\severity_results\pilot


In [17]:
# %% ── Cell 2: Imports ────────────────────────────────────────────────────────

import sys, os, warnings, time, pickle
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.use("Agg")
import seaborn as sns
from tqdm import tqdm
import scipy.sparse as sp
import scipy.stats as stats

import anndata as ad
import scanpy as sc
import gseapy as gp

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, r2_score,
    mean_squared_error, roc_auc_score, accuracy_score
)
from sklearn.neighbors import KNeighborsClassifier

try:
    import xgboost as xgb; HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed — skipping. pip install xgboost")

try:
    import shap; HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("SHAP not installed — skipping. pip install shap")

try:
    import umap; HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("umap-learn not installed — using scanpy UMAP. pip install umap-learn")

OUTDIR = Path(CONFIG["outdir"])
OUTDIR.mkdir(parents=True, exist_ok=True)
np.random.seed(CONFIG["random_seed"])
print(f"Output: {OUTDIR.resolve()}")

Output: C:\Users\ulago\Downloads\Applied-Machine-Learning-Final-Project\severity_results


In [18]:
# %% ── Cell 3: Load & preprocess ─────────────────────────────────────────────

def auto_col(adata, preferred):
    for p in preferred:
        for c in adata.obs.columns:
            if p.lower() in c.lower():
                return c
    for c in adata.obs.columns:
        if pd.api.types.is_categorical_dtype(adata.obs[c]) or \
           adata.obs[c].dtype == object:
            return c
    return None

print("\n[1/7] Loading data …")
adata = sc.read_h5ad(CONFIG["h5ad_path"])
print(f"      {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
print(f"      obs columns: {list(adata.obs.columns)}")

if adata.raw is not None:
    adata = adata.raw.to_adata()

if adata.X.max() > 50:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print("      Normalised + log1p")

GROUPBY = CONFIG["groupby"] or auto_col(
    adata, ["leiden","louvain","cell_type","celltype","cluster","annotation"])
SEV_COL = CONFIG["severity_col"] or auto_col(
    adata, ["severity","outcome","status","disease","hospitali",
            "icu","death","score","grade","who","clinical"])

print(f"      Groupby    : '{GROUPBY}'")
print(f"      Severity   : '{SEV_COL}'")

# ── HVG filtering ─────────────────────────────────────────────────────────────
if CONFIG["use_hvg"] and adata.n_vars > CONFIG["n_hvg"]:
    print(f"\n      Filtering to top {CONFIG['n_hvg']:,} HVGs …")
    sc.pp.highly_variable_genes(
        adata, n_top_genes=CONFIG["n_hvg"], flavor="seurat_v3")
    adata_full_genes = adata.copy()   # keep full for reference UMAP
    adata = adata[:, adata.var["highly_variable"]].copy()
    print(f"      HVG dataset: {adata.n_vars:,} genes retained")
else:
    adata_full_genes = adata.copy()


[1/7] Loading data …
      48,730 cells × 29,961 genes
      obs columns: ['nCount_RNA', 'nFeature_RNA', 'percent.mt', 'Variant_Group', 'Participant', 'SARSCoV2_PCR_Status', 'Vaccination_Status', 'WHO_Score_at_Peak', 'SingleCell_SARSCoV2_RNA_Status', 'Coarse_Annotation', 'Detailed_Annotation', 'Variant_Vax_Group']
      Normalised + log1p
      Groupby    : 'Coarse_Annotation'
      Severity   : 'WHO_Score_at_Peak'

      Filtering to top 5,000 HVGs …
      HVG dataset: 5,000 genes retained


In [19]:
# %% ── Cell 4: Stratified subsample ──────────────────────────────────────────
# This is the key cell for the pilot.
# Stratified sampling preserves the proportion of each cell type / cluster
# so the GO enrichment step sees a representative gene expression landscape.
# Without stratification, random sampling might drop rare cell types entirely,
# causing the mask to miss pathways specific to those populations.
 
def stratified_subsample(adata, fraction, stratify_col, seed):
    """
    Return a subsampled AnnData preserving per-stratum proportions.
 
    Parameters
    ----------
    adata        : full AnnData
    fraction     : float in (0, 1]
    stratify_col : obs column to stratify on; None = simple random
    seed         : int
    """
    rng = np.random.default_rng(seed)
    n_target = max(1, int(len(adata) * fraction))
 
    if stratify_col is None or stratify_col not in adata.obs.columns:
        # Simple random subsample
        idx = rng.choice(len(adata), size=n_target, replace=False)
        print(f"      Simple random subsample → {n_target:,} cells")
        return adata[idx].copy()
 
    # Stratified: sample proportionally from each stratum
    labels   = adata.obs[stratify_col].values
    strata   = np.unique(labels)
    selected = []
 
    for s in strata:
        stratum_idx = np.where(labels == s)[0]
        n_s = max(1, round(len(stratum_idx) * fraction))
        n_s = min(n_s, len(stratum_idx))          # cannot exceed stratum size
        chosen = rng.choice(stratum_idx, size=n_s, replace=False)
        selected.append(chosen)
 
    idx = np.concatenate(selected)
    rng.shuffle(idx)                               # shuffle to remove ordering
    print(f"      Stratified subsample (by '{stratify_col}') → {len(idx):,} cells")
 
    # Report per-stratum counts
    for s in strata:
        n_full = (labels == s).sum()
        n_samp = sum((labels[idx] == s))
        print(f"        {s:30s}  {n_full:>6,} → {n_samp:>5,}  "
              f"({n_samp/n_full*100:.1f}%)")
 
    return adata[idx].copy()
 
 
if CONFIG["PILOT_MODE"]:
    stratify_col = CONFIG["stratify_by"] or GROUPBY
    print(f"\n[2/7] Subsampling {int(CONFIG['pilot_fraction']*100)}% "
          f"(stratified by '{stratify_col}') …")
    adata = stratified_subsample(
        adata,
        fraction     = CONFIG["pilot_fraction"],
        stratify_col = stratify_col,
        seed         = CONFIG["pilot_seed"],
    )
else:
    print("\n[2/7] Full run — using all cells.")
    adata = adata.copy()
 
print(f"\n      Working dataset: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
 
# Save the subsample index so you can reproduce/inspect it later
pd.Series(adata.obs_names, name="cell_barcode").to_csv(
    OUTDIR / "pilot_cell_index.csv", index=False)
print(f"      Cell index saved → pilot_cell_index.csv")


[2/7] Subsampling 20% (stratified by 'Coarse_Annotation') …
      Stratified subsample (by 'Coarse_Annotation') → 9,747 cells
        B_Cell                             168 →    34  (20.2%)
        Basal                            1,073 →   215  (20.0%)
        Ciliated                        20,683 → 4,137  (20.0%)
        Dendritic                          416 →    83  (20.0%)
        Deuterosomal                       565 →   113  (20.0%)
        Goblet                             677 →   135  (19.9%)
        Ionocytes                          574 →   115  (20.0%)
        MT-high                          5,930 → 1,186  (20.0%)
        Macrophage                       2,119 →   424  (20.0%)
        SARSCoV2-high                    2,551 →   510  (20.0%)
        Secretory                        7,356 → 1,471  (20.0%)
        Squamous                         3,913 →   783  (20.0%)
        T_Cell                           2,705 →   541  (20.0%)

      Working dataset: 9,747 cells × 5,0

In [20]:
# %% ── Cell 4: Build GO pathway mask ─────────────────────────────────────────

def load_gmt(library, cache_dir):
    p = Path(cache_dir) / f"{library}.pkl"
    Path(cache_dir).mkdir(exist_ok=True)
    if p.exists():
        with open(p,"rb") as f: return pickle.load(f)
    print(f"      Downloading {library} …", end=" ")
    gs = gp.get_library(library, organism="Human")
    with open(p,"wb") as f: pickle.dump(gs, f)
    print(f"{len(gs):,} terms")
    return gs

print("\n[2/7] Building GO pathway mask …")
t0 = time.time()

all_gmt = {}
for lib in CONFIG["gmt_libraries"]:
    all_gmt.update(load_gmt(lib, CONFIG["gmt_cache_dir"]))
print(f"      Total GO terms: {len(all_gmt):,}")

gene_names  = list(adata.var_names)
gene_set    = set(gene_names)
gene_index  = {g: i for i, g in enumerate(gene_names)}

# Build mask AND store gene lists per pathway (needed for scoring)
pathway_names, pathway_gene_lists = [], []
for term, term_genes in all_gmt.items():
    overlap = [g for g in term_genes if g in gene_set]
    if CONFIG["min_genes_per_pathway"] <= len(overlap) <= CONFIG["max_genes_per_pathway"]:
        pathway_names.append(term)
        pathway_gene_lists.append(overlap)

print(f"      Pathways kept  : {len(pathway_names):,}")
print(f"      Built in       : {time.time()-t0:.1f}s")


[2/7] Building GO pathway mask …
      Total GO terms: 7,025
      Pathways kept  : 2,356
      Built in       : 0.1s


In [21]:
# %% ── Cell 5: Compute pathway activity scores → X matrix ─────────────────────
#
# For each cell and each pathway:
#   mean_z  : mean of z-scored gene expression across pathway members
#             (z-score normalises for gene-level mean/variance, making
#              pathways with different member counts comparable)
#   mean_raw: simple mean of log-normalised expression (faster)
#   pca1    : score = projection onto PC1 of pathway member genes
#
# Result: X  [n_cells × n_pathways]  — the numerical vector for ML

print("\n[3/7] Computing pathway activity scores …")
t0 = time.time()

X_raw = adata.X.toarray() if sp.issparse(adata.X) else adata.X.copy()
X_raw = X_raw.astype(np.float32)

method = CONFIG["score_method"]

if method in ("mean_z", "mean_raw"):
    if method == "mean_z":
        # Z-score each gene across cells first
        gene_mean = X_raw.mean(axis=0, keepdims=True)
        gene_std  = X_raw.std(axis=0, keepdims=True) + 1e-8
        X_scaled  = (X_raw - gene_mean) / gene_std
    else:
        X_scaled = X_raw

    # Score each pathway: mean over member genes
    n_cells    = X_raw.shape[0]
    n_pathways = len(pathway_names)
    X_pathway  = np.zeros((n_cells, n_pathways), dtype=np.float32)

    batch = 500   # process in batches to avoid OOM
    for start in tqdm(range(0, n_pathways, batch),
                      desc="      Scoring pathways"):
        end   = min(start + batch, n_pathways)
        for j, pw_genes in enumerate(pathway_gene_lists[start:end]):
            idx = [gene_index[g] for g in pw_genes]
            X_pathway[:, start + j] = X_scaled[:, idx].mean(axis=1)

elif method == "pca1":
    from sklearn.decomposition import PCA
    X_pathway = np.zeros((X_raw.shape[0], len(pathway_names)), dtype=np.float32)
    for j, pw_genes in enumerate(
            tqdm(pathway_gene_lists, desc="      PCA scoring")):
        idx   = [gene_index[g] for g in pw_genes]
        sub   = X_raw[:, idx]
        pca   = PCA(n_components=1, random_state=42)
        score = pca.fit_transform(sub).ravel()
        X_pathway[:, j] = score

elapsed = time.time() - t0
print(f"\n      X matrix shape : {X_pathway.shape}  "
      f"[{X_pathway.shape[0]:,} cells × {X_pathway.shape[1]:,} pathways]")
print(f"      Scoring time   : {elapsed:.1f}s")
print(f"      Value range    : [{X_pathway.min():.3f}, {X_pathway.max():.3f}]")
print(f"      Mean / std     : {X_pathway.mean():.3f} / {X_pathway.std():.3f}")

# Save X matrix
pw_df = pd.DataFrame(X_pathway, index=adata.obs_names, columns=pathway_names)
pw_df.to_csv(OUTDIR / "pathway_activity_matrix.csv")
print(f"      Saved → pathway_activity_matrix.csv")


[3/7] Computing pathway activity scores …


      Scoring pathways: 100%|██████████| 5/5 [00:04<00:00,  1.20it/s]



      X matrix shape : (9747, 2356)  [9,747 cells × 2,356 pathways]
      Scoring time   : 4.5s
      Value range    : [-0.615, 29.475]
      Mean / std     : -0.000 / 0.369
      Saved → pathway_activity_matrix.csv


In [22]:
# %% ── Cell 6: Prepare severity labels ───────────────────────────────────────

print("\n[4/7] Preparing severity labels …")

IS_CONTINUOUS = CONFIG["severity_type"] == "continuous"
HAS_SEVERITY  = SEV_COL is not None and SEV_COL in adata.obs.columns

if HAS_SEVERITY:
    raw_labels = adata.obs[SEV_COL].values
    if IS_CONTINUOUS:
        y = raw_labels.astype(np.float32)
        print(f"      Continuous severity: "
              f"mean={y.mean():.2f} std={y.std():.2f} "
              f"range=[{y.min():.2f}, {y.max():.2f}]")
        LABEL_ENCODER = None
    else:
        le = LabelEncoder()
        y  = le.fit_transform(raw_labels.astype(str)).astype(np.int64)
        LABEL_ENCODER = le
        print(f"      Classes: {dict(enumerate(le.classes_))}")
        print(f"      Distribution: {pd.Series(y).value_counts().to_dict()}")
else:
    print("      No severity column found — skipping ML, running UMAP only")
    HAS_SEVERITY = False
    y = None


[4/7] Preparing severity labels …
      Classes: {0: np.str_('0'), 1: np.str_('1'), 2: np.str_('2'), 3: np.str_('3'), 4: np.str_('4'), 5: np.str_('5'), 6: np.str_('6'), 7: np.str_('7'), 8: np.str_('8')}
      Distribution: {8: 2587, 0: 2285, 4: 1604, 5: 1044, 3: 937, 7: 630, 1: 305, 6: 221, 2: 134}


In [23]:
# %% ── Cell 7: Train/test split ──────────────────────────────────────────────

if HAS_SEVERITY:
    stratify = y if not IS_CONTINUOUS else None
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X_pathway, y, np.arange(len(y)),
        test_size    = CONFIG["test_fraction"],
        random_state = CONFIG["random_seed"],
        stratify     = stratify,
    )
    print(f"\n      Train: {len(X_train):,}  Test: {len(X_test):,}")


      Train: 7,797  Test: 1,950


In [24]:
# %% ── Cell 8: ML models ──────────────────────────────────────────────────────

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te,
                   continuous, label_encoder, outdir):
    """Fit, evaluate, and report one model."""
    print(f"\n  [{name}]")
    t0 = time.time()
    model.fit(X_tr, y_tr)
    print(f"    Trained in {time.time()-t0:.1f}s")

    y_pred = model.predict(X_te)

    results = {"model": name}

    if continuous:
        mse = mean_squared_error(y_te, y_pred)
        r2  = r2_score(y_te, y_pred)
        print(f"    MSE={mse:.4f}   R²={r2:.4f}")
        results.update({"mse": mse, "r2": r2})

        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(y_te, y_pred, alpha=0.4, s=8, color="#378ADD")
        lims = [min(y_te.min(), y_pred.min()), max(y_te.max(), y_pred.max())]
        ax.plot(lims, lims, "r--", lw=0.8)
        ax.set_xlabel("True severity"); ax.set_ylabel("Predicted")
        ax.set_title(f"{name}  R²={r2:.3f}")
        plt.tight_layout()
        safe = name.replace(" ", "_")
        fig.savefig(outdir / f"pred_scatter_{safe}.png", dpi=150,
                    bbox_inches="tight"); plt.close(fig)
    else:
        acc = accuracy_score(y_te, y_pred)
        print(f"    Accuracy={acc:.4f}")
        print(classification_report(
            y_te, y_pred,
            target_names=label_encoder.classes_ if label_encoder else None))
        results["accuracy"] = acc

        # Confusion matrix
        cm = confusion_matrix(y_te, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=label_encoder.classes_ if label_encoder else None,
                    yticklabels=label_encoder.classes_ if label_encoder else None)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(f"{name}  acc={acc:.3f}")
        plt.tight_layout()
        safe = name.replace(" ", "_")
        fig.savefig(outdir / f"confusion_{safe}.png", dpi=150,
                    bbox_inches="tight"); plt.close(fig)

        # AUC if binary
        if len(np.unique(y_te)) == 2 and hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_te)[:, 1]
            auc   = roc_auc_score(y_te, proba)
            print(f"    AUC={auc:.4f}")
            results["auc"] = auc

    return model, results


if HAS_SEVERITY:
    print("\n[5/7] Training ML models on pathway activity vectors …")
    all_results = []
    trained_models = {}

    # ── XGBoost ───────────────────────────────────────────────────────────
    if HAS_XGB:
        if IS_CONTINUOUS:
            xgb_model = xgb.XGBRegressor(
                n_estimators = CONFIG["xgb_n_estimators"],
                max_depth    = CONFIG["xgb_max_depth"],
                learning_rate= CONFIG["xgb_lr"],
                subsample    = 0.8,
                colsample_bytree=0.8,
                random_state = CONFIG["random_seed"],
                n_jobs       = -1,
                verbosity    = 0,
                #early_stopping_rounds = 20,
            )
        else:
            n_cls = len(np.unique(y_train))
            xgb_model = xgb.XGBClassifier(
                n_estimators    = CONFIG["xgb_n_estimators"],
                max_depth       = CONFIG["xgb_max_depth"],
                learning_rate   = CONFIG["xgb_lr"],
                subsample       = 0.8,
                colsample_bytree= 0.8,
                objective       = "multi:softprob" if n_cls > 2 else "binary:logistic",
                num_class       = n_cls if n_cls > 2 else None,
                random_state    = CONFIG["random_seed"],
                n_jobs          = -1,
                verbosity       = 0,
                eval_metric     = "mlogloss" if n_cls > 2 else "logloss",
                #early_stopping_rounds = 20,
            )
        m, r = evaluate_model("XGBoost", xgb_model,
                               X_train, y_train, X_test, y_test,
                               IS_CONTINUOUS, LABEL_ENCODER, OUTDIR)
        trained_models["XGBoost"] = m
        all_results.append(r)

    # ── Random Forest ─────────────────────────────────────────────────────
    rf_cls = RandomForestRegressor if IS_CONTINUOUS else RandomForestClassifier
    rf_model = rf_cls(
        n_estimators = CONFIG["rf_n_estimators"],
        max_depth    = 8,
        random_state = CONFIG["random_seed"],
        n_jobs       = -1,
        #early_stopping_rounds = 20,
    )
    m, r = evaluate_model("Random Forest", rf_model,
                           X_train, y_train, X_test, y_test,
                           IS_CONTINUOUS, LABEL_ENCODER, OUTDIR)
    trained_models["Random Forest"] = m
    all_results.append(r)

    # ── Elastic Net / Logistic Regression (linear, interpretable) ─────────
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_train)
    X_te_sc = scaler.transform(X_test)

    if IS_CONTINUOUS:
        lin_model = ElasticNet(
            alpha     = 0.01,
            l1_ratio  = CONFIG["enet_l1_ratio"],
            max_iter  = 2000,
            random_state = CONFIG["random_seed"],
        )
    else:
        lin_model = LogisticRegression(
            penalty  = "elasticnet",
            solver   = "saga",
            l1_ratio = CONFIG["enet_l1_ratio"],
            C        = 1.0,
            max_iter = 1000,
            random_state = CONFIG["random_seed"],
            n_jobs   = -1,
        )
    m, r = evaluate_model("Elastic Net", lin_model,
                           X_tr_sc, y_train, X_te_sc, y_test,
                           IS_CONTINUOUS, LABEL_ENCODER, OUTDIR)
    trained_models["Elastic Net"] = m
    all_results.append(r)

    # ── Summary table ─────────────────────────────────────────────────────
    results_df = pd.DataFrame(all_results).set_index("model")
    results_df.to_csv(OUTDIR / "model_comparison.csv")
    print(f"\n  Model comparison:\n{results_df.to_string()}")


[5/7] Training ML models on pathway activity vectors …

  [XGBoost]
    Trained in 281.0s
    Accuracy=0.5790
              precision    recall  f1-score   support

           0       0.55      0.76      0.64       457
           1       0.57      0.13      0.21        61
           2       0.76      0.48      0.59        27
           3       0.58      0.35      0.43       187
           4       0.58      0.47      0.52       321
           5       0.64      0.48      0.55       209
           6       1.00      0.09      0.17        44
           7       0.65      0.31      0.42       126
           8       0.58      0.77      0.66       518

    accuracy                           0.58      1950
   macro avg       0.66      0.43      0.47      1950
weighted avg       0.59      0.58      0.56      1950


  [Random Forest]
    Trained in 5.0s
    Accuracy=0.4533
              precision    recall  f1-score   support

           0       0.43      0.75      0.55       457
           1    

In [26]:
# %% ── Cell 9 replacement: SHAP pathway importance (fixed) ───────────────────

if HAS_SEVERITY and HAS_SHAP and HAS_XGB:
    print("\n  SHAP importance for XGBoost …")
    n_exp = min(CONFIG["n_shap_cells"], len(X_test))

    explainer   = shap.TreeExplainer(trained_models["XGBoost"])
    shap_values = explainer.shap_values(X_test[:n_exp])

    # Multi-class: shap_values is a list of [n_cells × n_features] arrays
    # Binary / regression: shap_values is a single array
    if isinstance(shap_values, list):
        sv = np.abs(np.stack(shap_values)).mean(axis=0)
    elif shap_values.ndim == 3:          # newer SHAP returns [n_cells, n_feat, n_class]
        sv = np.abs(shap_values).mean(axis=2)
    else:
        sv = np.abs(shap_values)

    mean_shap = sv.mean(axis=0)          # [n_pathways]
    top_n     = min(25, len(pathway_names))

    # ── THE FIX ──────────────────────────────────────────────────────────────
    # np.argsort returns numpy int64 values. Using them to index a plain Python
    # list raises "only integer scalar arrays can be converted to a scalar index"
    # in NumPy ≥ 1.24.  Convert to plain Python ints first.
    top_idx      = np.argsort(mean_shap)[::-1][:top_n]
    top_idx_list = [int(i) for i in top_idx]          # ← core fix

    pw_series  = pd.Series(pathway_names)             # index-safe Series
    top_labels = [pw_series.iloc[i][:65] for i in top_idx_list]
    top_scores = mean_shap[top_idx]                   # numpy fancy-index is fine

    fig, ax = plt.subplots(figsize=(10, 8))
    colors  = plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, top_n))
    ax.barh(top_labels, top_scores, color=colors)
    ax.invert_yaxis()
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title(f"Top {top_n} GO pathways driving severity prediction (XGBoost)")
    plt.tight_layout()
    fig.savefig(OUTDIR / "shap_top_pathways.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Ranked DataFrame — safe because pd.Series handles int indices cleanly
    shap_df = pd.DataFrame({
        "pathway":       pathway_names,
        "mean_abs_shap": mean_shap,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    shap_df["rank"] = shap_df.index + 1
    shap_df.to_csv(OUTDIR / "shap_pathway_importance.csv", index=False)

    print(f"  Top 5 pathways:")
    for _, row in shap_df.head(5).iterrows():
        print(f"    {int(row['rank']):>2}. {row['pathway'][:70]}"
              f"  SHAP={row['mean_abs_shap']:.4f}")
    print(f"  Saved shap_top_pathways.png")


  SHAP importance for XGBoost …
  Top 5 pathways:
     1. Negative Regulation Of Execution Phase Of Apoptosis (GO:1900118)  SHAP=0.1062
     2. Defense Response To Symbiont (GO:0140546)  SHAP=0.0890
     3. Regulation Of Neurotransmitter Receptor Activity (GO:0099601)  SHAP=0.0675
     4. Regulation Of Execution Phase Of Apoptosis (GO:1900117)  SHAP=0.0567
     5. Aldo-Keto Reductase (NADP) Activity (GO:0004033)  SHAP=0.0560
  Saved shap_top_pathways.png


In [27]:
# %% ── Cell 10: UMAP recreation study ────────────────────────────────────────
#
# Question: can pathway activity vectors (X_pathway) recreate the UMAP
# topology seen from raw gene expression?
#
# Method
# ──────
# 1. Compute reference UMAP from raw gene expression (PCA → UMAP)
# 2. Compute test UMAPs from:
#      a. X_pathway directly (cosine metric)
#      b. PCA of X_pathway (50 components) → UMAP
# 3. Quantify similarity:
#      a. Label transfer accuracy (kNN classifier trained on one UMAP,
#         tested on the other) — measures whether neighbourhoods align
#      b. Spearman correlation of pairwise distances (topology score)
#         on a random subsample of cells
#      c. Visual comparison of UMAP plots coloured by cluster + severity
 
print("\n[6/7] UMAP recreation study …")
 
# ── Reference: gene expression UMAP ──────────────────────────────────────────
print("  Computing reference UMAP from gene expression …")
adata_ref = adata.copy()
sc.pp.pca(adata_ref, n_comps=50)
sc.pp.neighbors(adata_ref, n_neighbors=CONFIG["umap_n_neighbors"],
                use_rep="X_pca")
sc.tl.umap(adata_ref, min_dist=CONFIG["umap_min_dist"])
UMAP_GENE = adata_ref.obsm["X_umap"].copy()
print(f"    Gene UMAP shape: {UMAP_GENE.shape}")



[6/7] UMAP recreation study …
  Computing reference UMAP from gene expression …
    Gene UMAP shape: (9747, 2)


In [28]:
# ── Test A: pathway activity UMAP (direct) ───────────────────────────────────
print("  Computing pathway-activity UMAP (direct) …")
adata_pw = adata.copy()
adata_pw.obsm["X_pathway"] = X_pathway
 
sc.pp.neighbors(adata_pw, n_neighbors=CONFIG["umap_n_neighbors"],
                use_rep="X_pathway",
                metric=CONFIG["umap_metric"])
sc.tl.umap(adata_pw, min_dist=CONFIG["umap_min_dist"])
UMAP_PW = adata_pw.obsm["X_umap"].copy()
print(f"    Pathway UMAP shape: {UMAP_PW.shape}")

  Computing pathway-activity UMAP (direct) …
    Pathway UMAP shape: (9747, 2)


In [29]:
# ── Test B: PCA of pathway activity → UMAP ───────────────────────────────────
print("  Computing pathway-PCA UMAP …")
from sklearn.decomposition import PCA as skPCA
n_pca_pw = min(50, X_pathway.shape[1] - 1)
pca_pw   = skPCA(n_components=n_pca_pw, random_state=CONFIG["random_seed"])
X_pca_pw = pca_pw.fit_transform(X_pathway)
 
adata_pw_pca = adata.copy()
adata_pw_pca.obsm["X_pathway_pca"] = X_pca_pw
sc.pp.neighbors(adata_pw_pca, n_neighbors=CONFIG["umap_n_neighbors"],
                use_rep="X_pathway_pca")
sc.tl.umap(adata_pw_pca, min_dist=CONFIG["umap_min_dist"])
UMAP_PW_PCA = adata_pw_pca.obsm["X_umap"].copy()
print(f"    Pathway-PCA UMAP shape: {UMAP_PW_PCA.shape}")
 

  Computing pathway-PCA UMAP …
    Pathway-PCA UMAP shape: (9747, 2)


In [30]:
# ── Quantitative comparison ───────────────────────────────────────────────────
print("\n  Quantifying UMAP similarity …")
 
def knn_label_transfer(embed_A, embed_B, labels, n_neighbors=15, seed=42):
    """
    Train kNN on embed_A with true labels, predict on embed_B.
    High accuracy = neighbourhood structure is preserved across embeddings.
    """
    knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric="euclidean")
    knn.fit(embed_A, labels)
    pred = knn.predict(embed_B)
    return accuracy_score(labels, pred)
 
 
def topology_score(embed_A, embed_B, n_sample=500, seed=42):
    """
    Spearman correlation between pairwise distances in embed_A and embed_B
    on a random cell subsample. Score of 1.0 = perfect topology preservation.
    """
    from sklearn.metrics import pairwise_distances
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(embed_A), size=min(n_sample, len(embed_A)),
                     replace=False)
    dA  = pairwise_distances(embed_A[idx], metric="euclidean").ravel()
    dB  = pairwise_distances(embed_B[idx], metric="euclidean").ravel()
    rho, pval = stats.spearmanr(dA, dB)
    return rho, pval
 
 
# Need cluster labels for kNN transfer
if GROUPBY and GROUPBY in adata.obs.columns:
    cluster_labels = LabelEncoder().fit_transform(
        adata.obs[GROUPBY].astype(str).values)
 
    acc_direct  = knn_label_transfer(UMAP_GENE, UMAP_PW,     cluster_labels)
    acc_pca     = knn_label_transfer(UMAP_GENE, UMAP_PW_PCA, cluster_labels)
    acc_self    = knn_label_transfer(UMAP_GENE, UMAP_GENE,   cluster_labels)
 
    print(f"\n  kNN label transfer accuracy (15-NN on UMAP coords):")
    print(f"    Gene UMAP → Gene UMAP (self, ceiling) : {acc_self:.3f}")
    print(f"    Gene UMAP → Pathway UMAP (direct)     : {acc_direct:.3f}")
    print(f"    Gene UMAP → Pathway-PCA UMAP           : {acc_pca:.3f}")
    pct_direct = acc_direct / acc_self * 100
    pct_pca    = acc_pca    / acc_self * 100
    print(f"\n    Pathway UMAP recreates {pct_direct:.1f}% of gene UMAP structure")
    print(f"    Pathway-PCA UMAP recreates {pct_pca:.1f}% of gene UMAP structure")
else:
    acc_direct = acc_pca = acc_self = None
    print("    No groupby column — skipping kNN transfer accuracy")
 
rho_direct, p_direct = topology_score(UMAP_GENE, UMAP_PW)
rho_pca,    p_pca    = topology_score(UMAP_GENE, UMAP_PW_PCA)
 
print(f"\n  Pairwise distance Spearman correlation (topology score):")
print(f"    Gene ↔ Pathway direct  : ρ={rho_direct:.3f}  p={p_direct:.2e}")
print(f"    Gene ↔ Pathway-PCA     : ρ={rho_pca:.3f}  p={p_pca:.2e}")
 
interpretation = (
    "strong"   if rho_direct > 0.7  else
    "moderate" if rho_direct > 0.4  else
    "weak"
)
print(f"\n  Interpretation: {interpretation} topology preservation")
print(f"  → Pathway activity vectors {'CAN' if rho_direct > 0.4 else 'CANNOT'} "
      f"meaningfully recreate gene UMAP structure")
 
# Save metrics
metrics = {
    "knn_transfer_self":          acc_self,
    "knn_transfer_pathway_direct":acc_direct,
    "knn_transfer_pathway_pca":   acc_pca,
    "topology_rho_direct":        rho_direct,
    "topology_p_direct":          p_direct,
    "topology_rho_pca":           rho_pca,
    "topology_p_pca":             p_pca,
    "n_pathways":                 len(pathway_names),
    "n_genes_covered":            sum(1 for pw in pathway_gene_lists
                                     if len(pw) > 0),
}
pd.Series(metrics).to_csv(OUTDIR / "umap_recreation_metrics.csv", header=False)


  Quantifying UMAP similarity …

  kNN label transfer accuracy (15-NN on UMAP coords):
    Gene UMAP → Gene UMAP (self, ceiling) : 0.848
    Gene UMAP → Pathway UMAP (direct)     : 0.425
    Gene UMAP → Pathway-PCA UMAP           : 0.006

    Pathway UMAP recreates 50.1% of gene UMAP structure
    Pathway-PCA UMAP recreates 0.7% of gene UMAP structure

  Pairwise distance Spearman correlation (topology score):
    Gene ↔ Pathway direct  : ρ=0.507  p=0.00e+00
    Gene ↔ Pathway-PCA     : ρ=0.582  p=0.00e+00

  Interpretation: moderate topology preservation
  → Pathway activity vectors CAN meaningfully recreate gene UMAP structure


In [31]:
# %% ── Cell 11: UMAP plots ────────────────────────────────────────────────────
 
print("\n[7/7] Plotting UMAPs …")
 
color_keys = [k for k in [GROUPBY, SEV_COL] if k and k in adata.obs.columns]
 
def plot_umap_comparison(umaps_dict, color_by, obs, outdir, suffix=""):
    """Plot multiple UMAPs side by side coloured by one obs column."""
    n    = len(umaps_dict)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    axes = np.atleast_1d(axes)
 
    labels   = obs[color_by].astype(str).values if color_by else None
    unique_l = np.unique(labels) if labels is not None else []
    cmap     = plt.cm.get_cmap("tab20", max(len(unique_l), 1))
 
    for ax, (title, coords) in zip(axes, umaps_dict.items()):
        if labels is not None:
            for i, lbl in enumerate(unique_l):
                m = labels == lbl
                ax.scatter(coords[m, 0], coords[m, 1],
                           s=2, alpha=0.5, color=cmap(i), label=lbl)
            ax.legend(markerscale=4, fontsize=7,
                      bbox_to_anchor=(1.01, 1), loc="upper left")
        else:
            ax.scatter(coords[:, 0], coords[:, 1], s=2, alpha=0.4, c="steelblue")
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
        ax.set_xticks([]); ax.set_yticks([])
 
    plt.suptitle(f"Coloured by: {color_by}", fontsize=12)
    plt.tight_layout()
    safe = (color_by or "none").replace(" ", "_")
    fname = f"umap_comparison_{safe}{suffix}.png"
    fig.savefig(outdir / fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    Saved {fname}")
 
 
umaps = {
    f"Gene expression\n(reference)":        UMAP_GENE,
    f"Pathway activity\n(direct)":           UMAP_PW,
    f"Pathway activity\n(PCA-compressed)":  UMAP_PW_PCA,
}
 
for ck in (color_keys if color_keys else [None]):
    plot_umap_comparison(umaps, ck, adata.obs, OUTDIR)
 
# ── Severity overlay specifically ─────────────────────────────────────────────
if SEV_COL and SEV_COL in adata.obs.columns:
    sev_vals = adata.obs[SEV_COL].values
    is_num   = np.issubdtype(sev_vals.dtype, np.number)
 
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (title, coords) in zip(axes, umaps.items()):
        if is_num:
            sc_obj = ax.scatter(coords[:, 0], coords[:, 1],
                                c=sev_vals.astype(float),
                                s=2, alpha=0.5, cmap="RdYlBu_r")
            plt.colorbar(sc_obj, ax=ax, label=SEV_COL)
        else:
            cats   = pd.Categorical(sev_vals)
            codes  = cats.codes
            cmap   = plt.cm.get_cmap("RdYlBu_r", cats.categories.size)
            sc_obj = ax.scatter(coords[:, 0], coords[:, 1],
                                c=codes, s=2, alpha=0.5, cmap=cmap)
            cbar   = plt.colorbar(sc_obj, ax=ax)
            cbar.set_ticks(range(cats.categories.size))
            cbar.set_ticklabels(cats.categories)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
        ax.set_xticks([]); ax.set_yticks([])
 
    plt.suptitle(f"Severity overlay: {SEV_COL}", fontsize=13)
    plt.tight_layout()
    fig.savefig(OUTDIR / "umap_severity_overlay.png",
                dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("    Saved umap_severity_overlay.png")
 
# ── Topology score visualisation ─────────────────────────────────────────────
from sklearn.metrics import pairwise_distances
rng_vis = np.random.default_rng(42)
n_vis   = min(300, len(UMAP_GENE))
idx_vis = rng_vis.choice(len(UMAP_GENE), n_vis, replace=False)
 
dG  = pairwise_distances(UMAP_GENE[idx_vis]).ravel()
dPW = pairwise_distances(UMAP_PW[idx_vis]).ravel()
 
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(dG, dPW, s=1, alpha=0.15, color="#378ADD")
ax.set_xlabel("Pairwise distance in gene UMAP")
ax.set_ylabel("Pairwise distance in pathway UMAP")
ax.set_title(f"Topology comparison\nρ={rho_direct:.3f}  (n={n_vis} cells)")
# Regression line
m_fit, b_fit = np.polyfit(dG, dPW, 1)
xl = np.array([dG.min(), dG.max()])
ax.plot(xl, m_fit * xl + b_fit, "r-", lw=1.2)
plt.tight_layout()
fig.savefig(OUTDIR / "topology_scatter.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("    Saved topology_scatter.png")


[7/7] Plotting UMAPs …
    Saved umap_comparison_Coarse_Annotation.png
    Saved umap_comparison_WHO_Score_at_Peak.png
    Saved umap_severity_overlay.png
    Saved topology_scatter.png


In [32]:
# %% ── Cell 12: Save everything + print report ────────────────────────────────
 
# Store embeddings in adata
adata.obsm["X_umap_gene"]       = UMAP_GENE
adata.obsm["X_umap_pathway"]    = UMAP_PW
adata.obsm["X_umap_pathway_pca"]= UMAP_PW_PCA
adata.obsm["X_pathway"]         = X_pathway
adata.write_h5ad(OUTDIR / "adata_with_pathway_umaps.h5ad")
 
print(f"\n{'='*62}")
print(f"  PIPELINE COMPLETE")
print(f"{'='*62}")
print(f"\n  Pathway vector (X matrix)")
print(f"    Shape          : {X_pathway.shape}")
print(f"    Pathways used  : {len(pathway_names):,}")
 
if HAS_SEVERITY:
    print(f"\n  Severity prediction")
    for r in all_results:
        key = "r2" if IS_CONTINUOUS else "accuracy"
        val = r.get(key, "n/a")
        print(f"    {r['model']:15s} : {key}={val:.3f}" if isinstance(val,float)
              else f"    {r['model']:15s} : {key}={val}")
 
print(f"\n  UMAP recreation")
print(f"    Topology score (ρ)           : {rho_direct:.3f}")
if acc_direct is not None:
    print(f"    kNN label transfer accuracy  : {acc_direct:.3f} "
          f"({pct_direct:.1f}% of gene UMAP ceiling)")
print(f"    Verdict: pathway vectors "
      f"{'SUCCESSFULLY RECREATE' if rho_direct > 0.5 else 'PARTIALLY RECREATE' if rho_direct > 0.3 else 'DO NOT RECREATE'} "
      f"gene UMAP topology")
 
print(f"\n  Outputs in {OUTDIR.resolve()}/")
print(f"    pathway_activity_matrix.csv      ← X matrix [cells × pathways]")
print(f"    model_comparison.csv             ← ML model metrics")
print(f"    shap_top_pathways.png/.csv       ← pathway importance")
print(f"    umap_comparison_*.png            ← side-by-side UMAPs")
print(f"    umap_severity_overlay.png        ← severity on each UMAP")
print(f"    topology_scatter.png             ← pairwise distance correlation")
print(f"    umap_recreation_metrics.csv      ← all quantitative scores")
print(f"    adata_with_pathway_umaps.h5ad    ← updated AnnData")


  PIPELINE COMPLETE

  Pathway vector (X matrix)
    Shape          : (9747, 2356)
    Pathways used  : 2,356

  Severity prediction
    XGBoost         : accuracy=0.579
    Random Forest   : accuracy=0.453
    Elastic Net     : accuracy=0.525

  UMAP recreation
    Topology score (ρ)           : 0.507
    kNN label transfer accuracy  : 0.425 (50.1% of gene UMAP ceiling)
    Verdict: pathway vectors SUCCESSFULLY RECREATE gene UMAP topology

  Outputs in C:\Users\ulago\Downloads\Applied-Machine-Learning-Final-Project\severity_results/
    pathway_activity_matrix.csv      ← X matrix [cells × pathways]
    model_comparison.csv             ← ML model metrics
    shap_top_pathways.png/.csv       ← pathway importance
    umap_comparison_*.png            ← side-by-side UMAPs
    umap_severity_overlay.png        ← severity on each UMAP
    topology_scatter.png             ← pairwise distance correlation
    umap_recreation_metrics.csv      ← all quantitative scores
    adata_with_pathway_umap